# Quering the SOSA graph

In [19]:
# imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [20]:
g = Graph()
## g.parse('/home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/sosa-graph.ttl', format='ttl')
g.parse('/home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/out/sosa.ttl', format='ttl')

<Graph identifier=N5d08a4cb54d842d2b712a17504202dd3 (<class 'rdflib.graph.Graph'>)>

## Quering the sosa graph

Check the journal to see all the queries to test on the SOSA graph

- Get all the resistance genes for a sample
- Rank the resistance genes found for a sample by the coverage percentage
- Get all the antibiotic, a specific strain is resistance to
- Get all the samples where a specific strain is present (filtering by mlst and by specie name)
- Get uniprot id of an antitbiotic and as well as it's molecular formula and a link to it's 3d structure
- Get the most spread resistance genes present in a certain geospacial area

In [3]:
## Filter samples by ST of the strains contained in them 

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT ?sampleId ?st 
    WHERE {
        ?samples a sosa:FeatureOfInterest ;
             rdf:type sio:001050 ;
             schema:identifier ?sampleId .
             
        ?observations sosa:hasFeatureOfInterest ?samples ;
             sosa:hasFeatureOfInterest ?strains .
        
        ?strains a aro:Strain ;
            abromics:has_st ?st . 
    }
    GROUP BY ?sampleId
""" 

res = g.query(query)

for row in res:
    print(f"{row.sampleId} - {row.st}")

0000 - 69
0001 - 48
0002 - 768
0003 - 10
0004 - 12
0005 - 38
0006 - 648
0008 - 10
0009 - 58


In [4]:
## Q1 of hackmd
## Get all the resistance gen for a sample 1

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?resGeneNames
    WHERE {
        ?sample a sosa:FeatureOfInterest .
        ?sample rdf:type sio:001050 ;
            schema:identifier "0000" .
        
        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Resistance gene"^^xsd:string .
        
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        ?observations sosa:hasFeatureOfInterest ?sample .
        ?observations sosa:hasResult ?results .
        ?results sosa:hasSimpleResult ?resGeneNames .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.resGeneNames}")

blaTEM-1A
tet(A)
sul1
qacE
dfrA1
aadA1


In [5]:
## Get all the resistance genes for a sample (with id = 0001) and rank them with their according to their coverage

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT DISTINCT ?geneNames ?geneCoverage 
    WHERE {
        ?sample a sosa:FeatureOfInterest ;
            schema:identifier "0000" .
        
        ?resGenesProperty a sosa:ObservableProperty ;
            rdfs:label "Resistance gene"^^xsd:string .
            
        ?observations sosa:hasObservableProperty ?resGenesProperty ;
            sosa:hasFeatureOfInterest ?genes ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasResult ?results .
            
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene ;
            rdfs:label ?geneNames .
            
        ?results sosa:hasSimpleResult ?geneCoverage .
    }
    ORDER BY DESC(?geneCoverage)
"""

res = g.query(query)

for row in res:
    print(f"{row.geneNames} - {row.geneCoverage}")

tet(A) - tet(A)
sul1 - sul1
qacE - qacE
dfrA1 - dfrA1
blaTEM-1A - blaTEM-1A
aadA1 - aadA1


In [6]:
# Q2 of hackmd (all samples)

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    PREFIX sio: <http://semanticscience.org/resource/>

    SELECT DISTINCT ?gene_name ?coverage ?s_id WHERE {
    
        ?sample schema:identifier ?s_id ;
            a sio:001050 .
        
        ?coverage_prop a sosa:ObservableProperty ;
                rdfs:label "Coverage %"^^xsd:string .
                
        ?observations sosa:hasObservableProperty ?coverage_prop ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasResult/sosa:hasSimpleResult ?coverage .
        
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene .
            
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?gene_name .
    } 
    ORDER BY DESC(?coverage) 
    LIMIT 10 # Top-10 most covered
"""

res = g.query(query)

for row in res:
    print(f"{row.gene_name} - {row.coverage} - {row.s_id}")

erm(B) - 100.13 - 0002
mph(A) - 100.11 - 0003
mph(A) - 100.11 - 0004
dfrA1 - 100.0 - 0000
dfrA1 - 100.0 - 0006
mcr-9 - 100.0 - 0004
fexA - 100.0 - 0002
catB3 - 100.0 - 0004
tet(L) - 100.0 - 0002
dfrG - 100.0 - 0002


In [8]:
## Q3: List the TOP-K antibiotic resistance genes by coverage in a geographical region of interest ##! Takes quite some time to execute

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?gene_name ?coverage ?s_id ?lat ?long ?alt 
    WHERE {
    
        ?location a sosa:FeatureOfInterest ;
            geo:lat "35.8648067"^^xsd:decimal ;
            geo:long "-120.6195831"^^xsd:decimal ;
            geo:alt "12.75"^^xsd:decimal .
    
        ?location geo:lat ?lat ;
            geo:long ?long ;
            geo:alt ?alt .
    
        ?sample schema:identifier ?s_id ;
            a sio:001050 .
        
        ?coverage_prop a sosa:ObservableProperty ;
                rdfs:label "Coverage %"^^xsd:string .
                
        ?observations sosa:hasObservableProperty ?coverage_prop ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasFeatureOfInterest ?location ;
            sosa:hasResult/sosa:hasSimpleResult ?coverage .
        
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene .
            
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?gene_name .
    }
    ORDER BY DESC(?coverage) 
    LIMIT 20 
"""

res = g.query(query)

for row in res:
    print(f"{row.gene_name} - {row.coverage} - {row.s_id} - {row.lat} - {row.long} - {row.alt}")

erm(B) - 100.13 - 0002 - 35.8648067 - -120.6195831 - 12.75
mph(A) - 100.11 - 0003 - 35.8648067 - -120.6195831 - 12.75
mph(A) - 100.11 - 0004 - 35.8648067 - -120.6195831 - 12.75
blaTEM-1A - 100.0 - 0000 - 35.8648067 - -120.6195831 - 12.75
dfrA1 - 100.0 - 0000 - 35.8648067 - -120.6195831 - 12.75
sitABCD - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
blaOXA-1 - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
catA1 - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
dfrA36 - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
tet(B) - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
aph(3')-Ia - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
sul1 - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
sul2 - 100.0 - 0001 - 35.8648067 - -120.6195831 - 12.75
ant(6)-Ia - 100.0 - 0002 - 35.8648067 - -120.6195831 - 12.75
optrA - 100.0 - 0002 - 35.8648067 - -120.6195831 - 12.75
dfrG - 100.0 - 0002 - 35.8648067 - -120.6195831 - 12.75
lsa(A) - 100.0 - 0002 - 35.8648067 - -120.6195831 - 12.75
tet(L) 

In [26]:
## Q3bis: List the TOP-K antibiotic resistance genes for a specific country

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?gene_name ?coverage
    WHERE {
    
        ?sample schema:identifier ?s_id ;
            a sio:001050 ;
            abromics:country "Belgium" .
        
        ?coverage_prop a sosa:ObservableProperty ;
                rdfs:label "Coverage %"^^xsd:string .
                
        ?observations sosa:hasObservableProperty ?coverage_prop ;
            sosa:hasFeatureOfInterest ?sample ;
            sosa:hasFeatureOfInterest ?location ;
            sosa:hasResult/sosa:hasSimpleResult ?coverage .
        
        ?genes a sosa:FeatureOfInterest ;
            a go:Gene .
            
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?genes rdfs:label ?gene_name .
    }
    ORDER BY DESC(?coverage) 
    LIMIT 20 
"""

res = g.query(query)

for row in res:
    print(f"{row.gene_name} - {row.coverage}")

erm(B) - 100.13
dfrA1 - 100.0
blaTEM-1A - 100.0
sul1 - 100.0
sitABCD - 100.0
tet(B) - 100.0
blaOXA-1 - 100.0
catA1 - 100.0
dfrA36 - 100.0
aph(3')-Ia - 100.0
sul2 - 100.0
optrA - 100.0
tet(M) - 100.0
fexA - 100.0
ant(6)-Ia - 100.0
dfrG - 100.0
lsa(A) - 100.0
tet(L) - 100.0
aph(3')-III - 100.0
blaCTX-M-55 - 100.0


In [7]:
## Q4: Common antibiotic resistance genes between two time points, for all available samples

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?resGeneName
    WHERE {

        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Resistance gene"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        
        ?observations sosa:hasResult/sosa:resultTime ?resTime .
        FILTER(?resTime > "2018-06-05T12:00:00Z"^^xsd:dateTime && ?resTime < "2019-07-01T12:00:00Z"^^xsd:dateTime).
        
        ?observations sosa:hasResult ?results .
        ?results sosa:hasSimpleResult ?resGeneName .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.resGeneName}")

sul2
sitABCD
tet(B)
blaCTX-M-15
dfrA17
aadA5
qacE
sul1
mph(A)
aac(3)-IIa
blaOXA-1
catB3
aac(6')-Ib-cr
aac(6')-Ib-cr
blaTEM-35
blaCTX-M-27
aph(3'')-Ib
aph(6)-Id
tet(B)


In [14]:
## Q6: Get all the antibiotics a specific specie (defined by its ncbi taxonomic id) is resistant to

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?antibiotics 
    WHERE {
        ?strain a aro:Strain ;
            a sosa:FeatureOfInterest .
        ?strain obo:RO_0002162 <http://purl.uniprot.org/taxonomy/562> . # Get the E.Coli taxon from ncbi
        ## doesn't work anymore -> this property should be given to a gene not a strain ?strain aro:confers_resistance_to_antibiotic ?antibiotics
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.antibiotics}")

In [22]:
## Q7: Find all the strain with the same antibiotic resistance gene found

## change "sul1" to "sul2" to showcase that the filtering process is working as the strain 1
## contains "sul1" but not "sul2" 

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?strains ?latitude ?longitude ?altitude
    WHERE {
        ?strains a aro:Strain .
            
        ?gene rdfs:label "sul1" .
        ?geneObservableProperty a sosa:ObservableProperty ;
            rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasFeatureOfInterest ?gene ;
            sosa:hasObservableProperty ?geneObservableProperty ;
            sosa:hasFeatureOfInterest ?strains ;
            sosa:hasFeatureOfInterest ?location .
        ?location geo:lat ?latitude ;
            geo:long ?longitude ;
            geo:alt ?altitude .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.strains} - {row.latitude} - {row.longitude} - {row.altitude}")